# Automotive Supply Chain — Exploratory Data Analysis (EDA)
## Supply Chain Intelligence & Business Analytics Platform

**Purpose:**  
This notebook walks through an exploratory analysis of the automotive supply chain dataset.  
It mirrors the analytical workflow a Supply Chain Analyst follows before building Power BI dashboards:
1. Understand the data structure
2. Identify data quality issues
3. Discover patterns and distributions
4. Calculate and validate KPIs
5. Identify the most important business findings

**How to run:**  
First run the setup scripts:
```bash
python scripts/data_generation/generate_datasets.py
python scripts/data_cleaning/01_data_cleaning_pipeline.py
```
Then open this notebook and run all cells.


In [ ]:
# ============================================================
# SETUP: Import libraries and set display options
# ============================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Display settings
pd.set_option('display.max_columns', 50)
pd.set_option('display.float_format', '{:.2f}'.format)
plt.rcParams['figure.figsize'] = (12, 5)
plt.rcParams['font.size'] = 11

# Color palette matching Power BI dashboard
COLORS = {
    'primary': '#1F3864',
    'accent': '#C9A84C',
    'success': '#107C10',
    'warning': '#FF8C00',
    'danger': '#D13438',
    'neutral': '#605E5C'
}

print('Libraries loaded successfully')
print('Ready to analyze automotive supply chain data')

In [ ]:
# ============================================================
# SECTION 1: LOAD DATA
# ============================================================

print('Loading cleaned datasets...')

shipments = pd.read_csv('data/processed/shipments_clean.csv', parse_dates=['shipment_date', 'actual_delivery_date'])
suppliers = pd.read_csv('data/processed/suppliers_clean.csv')
inventory = pd.read_csv('data/processed/inventory_clean.csv', parse_dates=['snapshot_date'])
production = pd.read_csv('data/processed/production_clean.csv', parse_dates=['production_date'])
sales = pd.read_csv('data/processed/sales_clean.csv', parse_dates=['order_date'])
warehouses = pd.read_csv('data/processed/warehouses_clean.csv')
plants = pd.read_csv('data/processed/plants_clean.csv')
products = pd.read_csv('data/processed/products_clean.csv')

print('\nDataset Summary:')
datasets = {
    'Shipments': shipments,
    'Suppliers': suppliers, 
    'Inventory': inventory,
    'Production': production,
    'Sales': sales,
    'Warehouses': warehouses,
    'Plants': plants,
    'Products': products
}

for name, df in datasets.items():
    print(f'  {name}: {len(df):,} rows × {len(df.columns)} columns')

total = sum(len(df) for df in datasets.values())
print(f'\n  TOTAL: {total:,} rows')

## SECTION 2: EXECUTIVE KPI DASHBOARD

Before diving deep, let's calculate the top-line KPIs that would appear on the Executive Dashboard.  
This is always the first thing a Supply Chain Analyst presents to leadership.

In [ ]:
# ============================================================
# EXECUTIVE KPIs — The numbers leadership sees every morning
# ============================================================

# On-Time Delivery %
otd_pct = shipments['is_on_time'].mean() * 100

# Inventory metrics (latest snapshot)
latest_inv = inventory.loc[inventory.groupby(['warehouse_id', 'product_id'])['snapshot_date'].idxmax()]
stockout_rate = (latest_inv['stock_quantity'] < latest_inv['safety_stock_level']).mean() * 100
total_inv_value = latest_inv['inventory_value_usd'].sum()

# Supplier reliability
avg_reliability = suppliers['reliability_score'].mean()

# Transportation cost
total_transport = shipments['transportation_cost_usd'].sum()

# Plant utilization
avg_utilization = production['utilization_rate_pct'].mean()

# Revenue
total_revenue = sales['revenue_usd'].sum()

# Forecast accuracy
forecast_acc = sales['forecast_accuracy_pct'].mean()

# Transportation cost ratio
tcr = total_transport / total_revenue * 100

# Display as dashboard
print('=' * 65)
print('  EXECUTIVE SUPPLY CHAIN KPI DASHBOARD')
print('  Automotive Supply Chain Intelligence Platform')
print('  Analysis Period: 2022-2024')
print('=' * 65)
print(f'\n  On-Time Delivery %:      {otd_pct:>8.1f}%   (Target: ≥95%)')
print(f'  Stockout Rate %:         {stockout_rate:>8.1f}%   (Target: <2%)')
print(f'  Inventory Value:         ${total_inv_value/1e9:>7.2f}B')
print(f'  Avg Supplier Reliability:{avg_reliability:>8.1f}    (Target: ≥85)')
print(f'  Total Transportation Cost:${total_transport/1e6:>6.1f}M')
print(f'  Transportation Cost Ratio:{tcr:>7.2f}%   (Target: <5%)')
print(f'  Plant Utilization %:     {avg_utilization:>8.1f}%   (Target: 80-90%)')
print(f'  Total Revenue:           ${total_revenue/1e9:>7.2f}B')
print(f'  Forecast Accuracy %:     {forecast_acc:>8.1f}%   (Target: ≥90%)')
print('=' * 65)

## SECTION 3: SUPPLIER PERFORMANCE ANALYSIS

**Business Question:** Which suppliers are causing the most delays?  
**Why it matters:** 80% of supply chain problems typically come from 20% of suppliers (Pareto principle).  
Identifying the underperformers is step 1 to fixing OTD.

In [ ]:
# ============================================================
# SUPPLIER OTD ANALYSIS
# ============================================================

# Calculate actual OTD per supplier from shipment transactions
supplier_otd = shipments.groupby('supplier_id').agg(
    total_shipments=('shipment_id', 'count'),
    on_time=('is_on_time', 'sum'),
    avg_delay=('delay_days', 'mean'),
    transport_cost=('transportation_cost_usd', 'sum')
).reset_index()

supplier_otd['otd_pct'] = supplier_otd['on_time'] / supplier_otd['total_shipments'] * 100

# Join with supplier names
supplier_otd = supplier_otd.merge(
    suppliers[['supplier_id', 'supplier_name', 'supplier_region', 'supplier_risk_score', 'risk_tier']],
    on='supplier_id', how='left'
)

# Bottom 10 performers
worst_suppliers = supplier_otd.nsmallest(10, 'otd_pct')

# Visualization
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

# Chart 1: Bottom 10 suppliers by OTD
colors = [COLORS['danger'] if x < 85 else COLORS['warning'] if x < 90 else COLORS['success'] 
          for x in worst_suppliers['otd_pct']]
ax1.barh(worst_suppliers['supplier_name'], worst_suppliers['otd_pct'], color=colors)
ax1.axvline(x=95, color='black', linestyle='--', linewidth=1.5, label='95% Target')
ax1.axvline(x=85, color=COLORS['danger'], linestyle=':', linewidth=1.5, label='85% Alert')
ax1.set_xlabel('On-Time Delivery %')
ax1.set_title('Bottom 10 Suppliers by On-Time Delivery\n(Red = Critical, Orange = Warning)', 
               fontweight='bold')
ax1.legend()
ax1.set_xlim(60, 100)

# Chart 2: OTD distribution histogram
ax2.hist(supplier_otd['otd_pct'], bins=15, color=COLORS['primary'], edgecolor='white', alpha=0.8)
ax2.axvline(x=95, color='black', linestyle='--', linewidth=2, label='95% Target')
ax2.axvline(x=supplier_otd['otd_pct'].mean(), color=COLORS['accent'], 
             linestyle='-', linewidth=2, label=f"Mean: {supplier_otd['otd_pct'].mean():.1f}%")
ax2.set_xlabel('OTD %')
ax2.set_ylabel('Number of Suppliers')
ax2.set_title('Distribution of Supplier OTD Performance', fontweight='bold')
ax2.legend()

plt.suptitle('Supplier On-Time Delivery Analysis', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('visuals/supplier_otd_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"\nSuppliers below 85% OTD: {(supplier_otd['otd_pct'] < 85).sum()} of {len(supplier_otd)}")
print(f"Suppliers below 95% target: {(supplier_otd['otd_pct'] < 95).sum()} of {len(supplier_otd)}")
print(f"\nWorst 5 performers:")
print(worst_suppliers[['supplier_name', 'supplier_region', 'otd_pct', 'avg_delay']].head(5).to_string(index=False))

In [ ]:
# ============================================================
# DELAY ANALYSIS BY REASON — ROOT CAUSE ANALYSIS
# Understanding WHY delays happen is more actionable than knowing IF they happen
# ============================================================

delayed = shipments[shipments['delay_days'] > 0].copy()

if 'delay_reason' in delayed.columns:
    delay_reasons = delayed['delay_reason'].value_counts()
    delay_reasons_pct = (delay_reasons / len(delayed) * 100).round(1)
    
    # Pareto chart — shows which causes to fix first for maximum impact
    cumulative_pct = delay_reasons_pct.cumsum()
    
    fig, ax1 = plt.subplots(figsize=(14, 6))
    ax2 = ax1.twinx()
    
    bars = ax1.bar(range(len(delay_reasons)), delay_reasons_pct, 
                   color=COLORS['primary'], alpha=0.8, edgecolor='white')
    ax2.plot(range(len(delay_reasons)), cumulative_pct, 
             color=COLORS['danger'], marker='o', linewidth=2, markersize=6)
    ax2.axhline(y=80, color='gray', linestyle='--', alpha=0.7, label='80% Pareto line')
    
    ax1.set_xticks(range(len(delay_reasons)))
    ax1.set_xticklabels(delay_reasons.index, rotation=45, ha='right')
    ax1.set_ylabel('% of Total Delays')
    ax2.set_ylabel('Cumulative %')
    ax2.set_ylim(0, 110)
    ax1.set_title('Delay Root Cause Pareto Analysis\n(Focus on left bars = 80% of delays)', 
                   fontweight='bold')
    ax2.legend(loc='center right')
    
    plt.tight_layout()
    plt.savefig('visuals/delay_pareto_analysis.png', dpi=150, bbox_inches='tight')
    plt.show()
    
    print('Delay Root Cause Breakdown:')
    for reason, pct in delay_reasons_pct.items():
        print(f'  {reason}: {pct:.1f}% of delays')

## SECTION 4: INVENTORY & WAREHOUSE ANALYSIS

**Business Question:** Which warehouses are at stockout risk?  
**Why it matters:** A single stockout of a critical component can halt a $1M+/day assembly line.

In [ ]:
# ============================================================
# INVENTORY HEALTH ANALYSIS
# ============================================================

# Use most recent snapshot for current state
latest_inv = inventory.loc[
    inventory.groupby(['warehouse_id', 'product_id'])['snapshot_date'].idxmax()
].copy()

# Warehouse-level summary
wh_summary = latest_inv.groupby('warehouse_id').agg(
    total_value=('inventory_value_usd', 'sum'),
    avg_dos=('days_of_supply', 'mean'),
    below_safety=('is_below_safety_stock', 'sum'),
    total_skus=('product_id', 'nunique')
).reset_index()

wh_summary['stockout_rate'] = wh_summary['below_safety'] / wh_summary['total_skus'] * 100
wh_summary = wh_summary.merge(
    warehouses[['warehouse_id', 'warehouse_name', 'region', 'current_utilization_pct']],
    on='warehouse_id', how='left'
)

# Visualization
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

# Chart 1: Stockout rate by warehouse
wh_sorted = wh_summary.sort_values('stockout_rate', ascending=False)
colors = [COLORS['danger'] if x > 20 else COLORS['warning'] if x > 10 else COLORS['success'] 
          for x in wh_sorted['stockout_rate']]
ax1.barh(wh_sorted['warehouse_id'], wh_sorted['stockout_rate'], color=colors)
ax1.axvline(x=2, color='black', linestyle='--', label='2% Target')
ax1.set_xlabel('Stockout Rate %')
ax1.set_title('Stockout Rate by Warehouse\n(Red >20%, Orange >10%, Green <10%)', fontweight='bold')
ax1.legend()

# Chart 2: Avg days of supply by warehouse
ax2.barh(wh_sorted['warehouse_id'], wh_sorted['avg_dos'], 
          color=[COLORS['danger'] if x < 7 else COLORS['warning'] if x < 14 else COLORS['success'] 
                 for x in wh_sorted['avg_dos']])
ax2.axvline(x=7, color=COLORS['danger'], linestyle='--', label='7-day Critical')
ax2.axvline(x=14, color=COLORS['warning'], linestyle=':', label='14-day Warning')
ax2.set_xlabel('Average Days of Supply')
ax2.set_title('Avg Days of Supply by Warehouse\n(Red <7 days = Emergency)', fontweight='bold')
ax2.legend()

plt.suptitle('Inventory Health Dashboard', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('visuals/inventory_health_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"\nCRITICAL warehouses (stockout rate >20%): {(wh_summary['stockout_rate'] > 20).sum()}")
print(f"HIGH RISK warehouses (stockout rate >10%): {(wh_summary['stockout_rate'] > 10).sum()}")
print(f"Overall stockout rate: {latest_inv['is_below_safety_stock'].mean()*100:.1f}%")

## SECTION 5: OTD TREND ANALYSIS (Time Series)

**Business Question:** Is our on-time delivery improving or getting worse?  
**Why it matters:** Trend direction matters as much as absolute level. A 90% OTD that's improving is better than 92% that's declining.

In [ ]:
# ============================================================
# OTD TREND OVER TIME (24 months)
# ============================================================

# Monthly OTD trend
shipments['month_period'] = shipments['shipment_date'].dt.to_period('M')
monthly_otd = shipments.groupby('month_period').agg(
    otd_pct=('is_on_time', lambda x: x.mean() * 100),
    shipment_count=('shipment_id', 'count'),
    transport_cost=('transportation_cost_usd', 'sum')
).reset_index()
monthly_otd['month_str'] = monthly_otd['month_period'].astype(str)

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(16, 10))

# Chart 1: OTD trend
x = range(len(monthly_otd))
ax1.plot(x, monthly_otd['otd_pct'], color=COLORS['primary'], linewidth=2.5, marker='o', markersize=5)
ax1.axhline(y=95, color=COLORS['success'], linestyle='--', linewidth=1.5, label='95% Target')
ax1.axhline(y=85, color=COLORS['danger'], linestyle=':', linewidth=1.5, label='85% Alert')
ax1.fill_between(x, monthly_otd['otd_pct'], 85, 
                  where=[v < 85 for v in monthly_otd['otd_pct']], 
                  alpha=0.3, color=COLORS['danger'], label='Below Alert')
ax1.set_xticks(x[::3])  # Show every 3rd month
ax1.set_xticklabels([monthly_otd['month_str'].iloc[i] for i in x[::3]], rotation=45)
ax1.set_ylabel('On-Time Delivery %')
ax1.set_title('On-Time Delivery % — 36-Month Trend (2022-2024)', fontweight='bold')
ax1.legend()
ax1.set_ylim(70, 100)
ax1.grid(axis='y', alpha=0.3)

# Add annotation for COVID/chip shortage period
ax1.annotate('Supply Chain\nDisruptions\n(Chip Shortage)', 
              xy=(3, monthly_otd['otd_pct'].iloc[3] if len(monthly_otd) > 3 else 80),
              xytext=(6, 74),
              arrowprops=dict(arrowstyle='->', color=COLORS['danger']),
              fontsize=9, color=COLORS['danger'])

# Chart 2: Monthly shipment volume
ax2.bar(x, monthly_otd['shipment_count'], color=COLORS['primary'], alpha=0.7)
ax2.set_xticks(x[::3])
ax2.set_xticklabels([monthly_otd['month_str'].iloc[i] for i in x[::3]], rotation=45)
ax2.set_ylabel('Number of Shipments')
ax2.set_title('Monthly Shipment Volume', fontweight='bold')
ax2.grid(axis='y', alpha=0.3)

plt.suptitle('Supply Chain Performance Trend Analysis', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('visuals/otd_trend_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

# Year-over-year comparison
shipments['year'] = shipments['shipment_date'].dt.year
yoy_otd = shipments.groupby('year')['is_on_time'].mean() * 100
print('\nYear-over-Year OTD Performance:')
for yr, otd in yoy_otd.items():
    target_gap = otd - 95
    status = '✓' if otd >= 95 else '⚠' if otd >= 85 else '✗'
    print(f'  {yr}: {otd:.1f}%  {status}  ({target_gap:+.1f}pp vs 95% target)')

## SECTION 6: EV DEMAND TREND ANALYSIS

**Business Question:** Is EV demand growing? Is it outpacing our supply chain readiness?  
**Why it matters:** EV growth is the #1 strategic shift in automotive. Supply chains that don't adapt will be left behind.

In [ ]:
# ============================================================
# EV VS ICE DEMAND TREND ANALYSIS
# ============================================================

# Annual sales by vehicle category and powertrain
sales['year'] = sales['order_date'].dt.year
sales['is_ev'] = sales['vehicle_category'].str.contains('Electric|EV', case=False, na=False)

annual_by_type = sales.groupby(['year', 'vehicle_category']).agg(
    units=('units_sold', 'sum'),
    revenue=('revenue_usd', 'sum')
).reset_index()

# EV vs ICE comparison
ev_vs_ice = sales.groupby(['year', 'is_ev']).agg(
    units=('units_sold', 'sum'),
    revenue=('revenue_usd', 'sum')
).reset_index()
ev_vs_ice['type'] = ev_vs_ice['is_ev'].map({True: 'Electric / Hybrid', False: 'ICE / Combustion'})

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

# Chart 1: EV vs ICE units over time
for ev_type, color in [('Electric / Hybrid', COLORS['success']), ('ICE / Combustion', COLORS['neutral'])]:
    data = ev_vs_ice[ev_vs_ice['type'] == ev_type]
    ax1.plot(data['year'], data['units'], marker='o', linewidth=2.5,
              color=color, label=ev_type, markersize=8)
    # Add value labels
    for _, row in data.iterrows():
        ax1.annotate(f"{row['units']:,.0f}", (row['year'], row['units']), 
                      textcoords='offset points', xytext=(0, 10), ha='center', fontsize=9)

ax1.set_xlabel('Year')
ax1.set_ylabel('Units Sold')
ax1.set_title('EV vs ICE Vehicle Sales Trend\n(EV growth accelerating)', fontweight='bold')
ax1.legend()
ax1.grid(axis='y', alpha=0.3)
ax1.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:,.0f}'))

# Chart 2: Forecast vs Actual accuracy
monthly_forecast = sales.groupby(sales['order_date'].dt.to_period('Q')).agg(
    actual=('units_sold', 'sum'),
    forecast=('forecasted_demand', 'sum')
).reset_index()
monthly_forecast['period_str'] = monthly_forecast['order_date'].astype(str)

x = range(len(monthly_forecast))
ax2.plot(x, monthly_forecast['actual'], color=COLORS['primary'], linewidth=2.5, label='Actual Demand', marker='o', markersize=5)
ax2.plot(x, monthly_forecast['forecast'], color=COLORS['accent'], linewidth=2, 
          linestyle='--', label='Forecasted Demand', marker='s', markersize=5)
ax2.fill_between(x, monthly_forecast['actual'], monthly_forecast['forecast'], alpha=0.2, 
                  color=COLORS['danger'], label='Forecast Error')
ax2.set_xticks(x[::2])
ax2.set_xticklabels([monthly_forecast['period_str'].iloc[i] for i in x[::2]], rotation=45, fontsize=8)
ax2.set_ylabel('Units')
ax2.set_title('Forecast vs Actual Demand\n(Systematic under-forecast in recent quarters)', fontweight='bold')
ax2.legend()
ax2.grid(axis='y', alpha=0.3)

plt.suptitle('Demand & Sales Analytics', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('visuals/demand_trend_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

# EV growth calculation
ev_data = ev_vs_ice[ev_vs_ice['type'] == 'Electric / Hybrid'].sort_values('year')
if len(ev_data) >= 2:
    ev_growth = (ev_data['units'].iloc[-1] - ev_data['units'].iloc[-2]) / ev_data['units'].iloc[-2] * 100
    print(f"\nEV/Hybrid YoY growth (most recent year): {ev_growth:.1f}%")
    
ice_data = ev_vs_ice[ev_vs_ice['type'] == 'ICE / Combustion'].sort_values('year')
if len(ice_data) >= 2:
    ice_growth = (ice_data['units'].iloc[-1] - ice_data['units'].iloc[-2]) / ice_data['units'].iloc[-2] * 100
    print(f"ICE YoY growth (most recent year): {ice_growth:.1f}%")

## SECTION 7: MANUFACTURING ANALYSIS

**Business Question:** Which plants are underperforming and what's causing downtime?  
**Why it matters:** Every hour of unplanned downtime costs automotive plants $1M+ in lost production.

In [ ]:
# ============================================================
# MANUFACTURING PERFORMANCE ANALYSIS
# ============================================================

# Plant-level summary
plant_summary = production.groupby('plant_id').agg(
    avg_utilization=('utilization_rate_pct', 'mean'),
    total_downtime=('downtime_hours', 'sum'),
    defect_rate=('defect_rate_pct', 'mean'),
    total_produced=('actual_production_units', 'sum'),
    total_planned=('planned_production_units', 'sum')
).reset_index()

plant_summary = plant_summary.merge(
    plants[['plant_id', 'plant_name', 'oem_company', 'state']],
    on='plant_id', how='left'
)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 7))

# Chart 1: Plant utilization
plant_sorted = plant_summary.sort_values('avg_utilization', ascending=True)
colors = [COLORS['danger'] if x < 70 else COLORS['warning'] if x < 80 else 
           COLORS['success'] if x <= 90 else COLORS['warning'] for x in plant_sorted['avg_utilization']]
bars = ax1.barh(plant_sorted['plant_id'], plant_sorted['avg_utilization'], color=colors)
ax1.axvline(x=80, color='black', linestyle='--', linewidth=1.5, label='80% Lower Target')
ax1.axvline(x=90, color='black', linestyle=':', linewidth=1.5, label='90% Upper Target')
ax1.set_xlabel('Plant Utilization %')
ax1.set_title('Plant Utilization %\n(Green = Optimal 80-90%, Red = Critical <70%)', fontweight='bold')
ax1.legend()
ax1.set_xlim(0, 110)

# Chart 2: Downtime by plant
plant_dt_sorted = plant_summary.sort_values('total_downtime', ascending=False)
ax2.bar(range(len(plant_dt_sorted)), plant_dt_sorted['total_downtime'], 
         color=COLORS['primary'], alpha=0.8)
ax2.set_xticks(range(len(plant_dt_sorted)))
ax2.set_xticklabels(plant_dt_sorted['plant_id'], rotation=45, ha='right')
ax2.set_ylabel('Total Downtime Hours (2022-2024)')
ax2.set_title('Total Downtime Hours by Plant\n(Prioritize highest bars)', fontweight='bold')
ax2.grid(axis='y', alpha=0.3)

plt.suptitle('Manufacturing Operations Dashboard', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('visuals/manufacturing_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

print('\nPlant Performance Summary:')
print(plant_summary[['plant_id', 'avg_utilization', 'total_downtime', 'defect_rate']]
      .sort_values('avg_utilization')
      .to_string(index=False))

## SECTION 8: KEY FINDINGS & RECOMMENDATIONS

This section summarizes the most important analytical findings.  
In a real project, this would form the basis of the executive presentation.

In [ ]:
# ============================================================
# EXECUTIVE SUMMARY: KEY FINDINGS
# ============================================================

print('='*65)
print('  KEY ANALYTICAL FINDINGS')
print('  Automotive Supply Chain Intelligence Platform')
print('='*65)

# Finding 1: Supplier performance
low_otd = (supplier_otd['otd_pct'] < 85).sum()
total_sup = len(supplier_otd)
print(f'\n1. SUPPLIER PERFORMANCE')
print(f'   {low_otd} of {total_sup} suppliers ({low_otd/total_sup*100:.0f}%) below 85% OTD target')
print(f'   Overall OTD: {otd_pct:.1f}% vs 95% target (gap: {95-otd_pct:.1f}pp)')
print(f'   Action: Issue SCAPs to {low_otd} underperforming suppliers')

# Finding 2: Inventory
critical_inv = (latest_inv['days_of_supply'] < 7).sum()
print(f'\n2. INVENTORY RISK')
print(f'   {critical_inv} SKU-locations below 7-day critical threshold')
print(f'   Overall stockout rate: {stockout_rate:.1f}% vs 2% target')
print(f'   Action: Emergency replenishment for critical items')

# Finding 3: Manufacturing
low_util_plants = (plant_summary['avg_utilization'] < 75).sum()
print(f'\n3. MANUFACTURING EFFICIENCY')
print(f'   {low_util_plants} plants below 75% utilization threshold')
print(f'   Avg plant utilization: {avg_utilization:.1f}% (target: 80-90%)')
print(f'   Action: Downtime root cause analysis and maintenance investment')

# Finding 4: EV demand
print(f'\n4. EV DEMAND ACCELERATION')
print(f'   Forecast accuracy: {forecast_acc:.1f}% (target: ≥90%)')
print(f'   Systematic under-forecast suggests EV adoption model needs update')
print(f'   Action: Recalibrate demand model with updated EV adoption curves')

# Finding 5: Logistics
print(f'\n5. LOGISTICS COSTS')
print(f'   Transportation cost ratio: {tcr:.2f}% of revenue (target: <5%)')
print(f'   Total freight spend: ${total_transport/1e6:.1f}M')
print(f'   Action: Carrier consolidation and route optimization program')

print('\n' + '='*65)
print('  END OF EXPLORATORY DATA ANALYSIS')
print('  Next step: Power BI Dashboard Development')
print('  Reference: docs/dashboard_docs/01_power_bi_dashboard_guide.md')
print('='*65)